# Automated Attendance Sheet Generator

**Required inputs (every run):**
- Exam schedule PDF
- Student master list PDF (grade + section wise names)
- Attendance sheet template PDF

This notebook parses the PDFs and produces one combined PDF per grade (9-12).


In [1]:
from __future__ import annotations

import re
from datetime import date
from typing import List, Optional

from reportlab.pdfbase import pdfmetrics


def ordinal(n: int) -> str:
    if 10 <= n % 100 <= 20:
        suffix = "th"
    else:
        suffix = {1: "st", 2: "nd", 3: "rd"}.get(n % 10, "th")
    return f"{n}{suffix}"


def format_date_with_day(d: date, day_name: Optional[str] = None) -> str:
    day = day_name or d.strftime('%A')
    day_map = {
        'Mon': 'Monday',
        'Tue': 'Tuesday',
        'Tues': 'Tuesday',
        'Wed': 'Wednesday',
        'Thu': 'Thursday',
        'Thur': 'Thursday',
        'Thurs': 'Thursday',
        'Fri': 'Friday',
        'Sat': 'Saturday',
        'Sun': 'Sunday',
    }
    day = day_map.get(day, day)
    return f"{ordinal(d.day)} {d.strftime('%B %Y')}, {day}"


def normalize_time_range(raw: str) -> str:
    if not raw:
        return ""

    text = raw.strip().strip('[]()')
    text = text.replace('–', '-').replace('—', '-').replace('?', '-')
    time_re = re.compile(r"(\d{1,2})[:.](\d{2})\s*(a\.?m\.?|p\.?m\.?)", re.I)
    parts = time_re.findall(text)

    def fmt(part):
        h, m, ap = part
        ap_norm = 'am' if ap.lower().startswith('a') else 'pm'
        return f"{int(h)}:{m} {ap_norm}"

    if len(parts) >= 2:
        return f"{fmt(parts[0])} - {fmt(parts[1])}"
    if len(parts) == 1:
        return fmt(parts[0])
    return text


def join_subjects(subjects: List[str]) -> str:
    subjects = [s.strip() for s in subjects if s and s.strip()]
    if not subjects:
        return ""
    if len(subjects) == 1:
        return subjects[0]
    if len(subjects) == 2:
        return f"{subjects[0]} & {subjects[1]}"
    return ", ".join(subjects[:-1]) + f" & {subjects[-1]}"


def wrap_text(text: str, font_name: str, font_size: float, max_width: float) -> List[str]:
    words = text.split()
    if not words:
        return []
    lines: List[str] = []
    current: List[str] = []
    for word in words:
        trial = " ".join(current + [word])
        if pdfmetrics.stringWidth(trial, font_name, font_size) <= max_width or not current:
            current.append(word)
        else:
            lines.append(" ".join(current))
            current = [word]
    if current:
        lines.append(" ".join(current))
    return lines


def split_multi_name_line(line: str) -> List[str]:
    parts = [p.strip() for p in re.split(r"\s{2,}", line) if p.strip()]
    return parts if parts else [line.strip()]


In [2]:

from dataclasses import dataclass
from pathlib import Path
from statistics import median
from typing import Dict, List, Optional, Tuple

import pdfplumber
from pypdf import PdfReader


@dataclass
class LineSeg:
    x0: float
    y0: float
    x1: float
    y1: float
    width: float


@dataclass
class Rect:
    x0: float
    y0: float
    x1: float
    y1: float
    fill_rgb: Tuple[float, float, float]


@dataclass
class TemplateFonts:
    gill_regular: str
    gill_bold: str
    georgia_bold: str
    size: float


@dataclass
class TemplateLayout:
    page_width: float
    page_height: float
    lines: List[LineSeg]
    rects: List[Rect]
    fonts: TemplateFonts
    anchors: Dict[str, float | List[float]]


def _dominant_font(word, chars) -> Tuple[str, float]:
    font_counts: Dict[str, int] = {}
    size_counts: Dict[float, int] = {}
    for ch in chars:
        if (
            ch["x0"] >= word["x0"] - 0.5
            and ch["x1"] <= word["x1"] + 0.5
            and ch["top"] >= word["top"] - 0.5
            and ch["bottom"] <= word["bottom"] + 0.5
        ):
            font_counts[ch["fontname"]] = font_counts.get(ch["fontname"], 0) + 1
            size = round(ch["size"], 2)
            size_counts[size] = size_counts.get(size, 0) + 1
    if not font_counts:
        return ("", 12.0)
    font = max(font_counts.items(), key=lambda x: x[1])[0]
    size = max(size_counts.items(), key=lambda x: x[1])[0] if size_counts else 12.0
    base_font = font.split("+")[-1]
    return base_font, float(size)


def _group_words_by_line(words, tolerance: float = 1.5):
    lines: List[Dict] = []
    for w in sorted(words, key=lambda x: x["top"]):
        placed = False
        for line in lines:
            if abs(line["top"] - w["top"]) <= tolerance:
                line["words"].append(w)
                placed = True
                break
        if not placed:
            lines.append({"top": w["top"], "words": [w]})
    for line in lines:
        line["words"].sort(key=lambda x: x["x0"])
        line["text"] = " ".join(w["text"] for w in line["words"]).strip()
    return lines


def extract_fonts_from_template(template_pdf: str | Path, output_dir: str | Path) -> Dict[str, Path]:
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    reader = PdfReader(str(template_pdf))
    extracted: Dict[str, Path] = {}
    for page in reader.pages:
        fonts = (page.get("/Resources") or {}).get("/Font") or {}
        for font_ref in fonts.values():
            font = font_ref.get_object()
            descendants = font.get("/DescendantFonts") or []
            for df in descendants:
                df_obj = df.get_object()
                base_name = df_obj.get("/BaseFont") or font.get("/BaseFont")
                if not base_name:
                    continue
                base_name = str(base_name)
                if base_name.startswith("/"):
                    base_name = base_name[1:]
                clean_name = base_name.split("+")[-1]
                if clean_name in extracted:
                    continue
                desc = df_obj.get("/FontDescriptor")
                if not desc:
                    continue
                desc_obj = desc.get_object()
                ff2 = desc_obj.get("/FontFile2")
                if ff2:
                    data = ff2.get_data()
                    out_path = output_dir / f"{clean_name}.ttf"
                    out_path.write_bytes(data)
                    extracted[clean_name] = out_path
    return extracted


def analyze_template(template_pdf: str | Path) -> TemplateLayout:
    with pdfplumber.open(template_pdf) as pdf:
        page = pdf.pages[0]
        page_width = page.width
        page_height = page.height
        words = page.extract_words()
        chars = page.chars

        lines_raw = []
        for ln in page.lines:
            lines_raw.append(
                LineSeg(
                    x0=float(ln["x0"]),
                    y0=float(ln["y0"]),
                    x1=float(ln["x1"]),
                    y1=float(ln["y1"]),
                    width=float(ln.get("linewidth") or 1.0),
                )
            )

        rects_raw: List[Rect] = []
        for r in page.rects:
            fill = r.get("non_stroking_color")
            if not fill:
                continue
            # Skip full-page white background
            if (
                abs(r["x0"]) < 1e-3
                and abs(r["y0"]) < 1e-3
                and abs(r["x1"] - page_width) < 1e-3
                and abs(r["y1"] - page_height) < 1e-3
            ):
                continue
            rects_raw.append(
                Rect(
                    x0=float(r["x0"]),
                    y0=float(r["y0"]),
                    x1=float(r["x1"]),
                    y1=float(r["y1"]),
                    fill_rgb=tuple(float(x) for x in (fill if isinstance(fill, (list, tuple)) else (0, 0, 0))),
                )
            )

        grouped_lines = _group_words_by_line(words)

        def words_on_line(anchor_word, tolerance: float = 1.5):
            return sorted(
                [w for w in words if abs(w["top"] - anchor_word["top"]) <= tolerance],
                key=lambda w: w["x0"],
            )

        def find_line(predicate, candidates=None) -> Optional[dict]:
            lines = candidates or grouped_lines
            for line in lines:
                if predicate(line):
                    return line
            return None

        grade_line = find_line(lambda l: "Grade" in l["text"])
        date_line = find_line(lambda l: "Date" in l["text"] and "Day" in l["text"])
        subject_line = find_line(lambda l: l["text"].startswith("Subject"))
        header_line = find_line(lambda l: "Sl." in l["text"] and "Name" in l["text"])

        # Fallback: locate key words directly if line grouping fails
        word_by_text = {}
        for w in words:
            word_by_text.setdefault(w["text"], []).append(w)

        grade_word = None
        if grade_line:
            grade_word = next(w for w in grade_line["words"] if w["text"] == "Grade")
        else:
            grade_word = (word_by_text.get("Grade") or [None])[0]

        if not grade_word:
            raise RuntimeError("Could not locate 'Grade' in template PDF.")

        if not date_line:
            date_word = (word_by_text.get("Date") or [None])[0]
            day_word = (word_by_text.get("Day:") or word_by_text.get("Day") or [None])[0]
            if date_word:
                date_line = {"words": words_on_line(date_word), "text": "", "top": date_word["top"]}
            elif day_word:
                date_line = {"words": words_on_line(day_word), "text": ""}

        if not subject_line:
            subject_word = (word_by_text.get("Subject:") or word_by_text.get("Subject") or [None])[0]
            if subject_word:
                subject_line = {"words": words_on_line(subject_word), "text": "", "top": subject_word["top"]}

        if not header_line:
            sl_word_fallback = (word_by_text.get("Sl.") or word_by_text.get("Sl") or [None])[0]
            if sl_word_fallback:
                header_line = {"words": words_on_line(sl_word_fallback), "text": "", "top": sl_word_fallback["top"]}

        if not (date_line and subject_line and header_line):
            raise RuntimeError("Could not locate key header lines in template PDF.")

        # Font mapping
        gill_bold, font_size = _dominant_font(grade_word, chars)

        num_word = next((w for w in words if w["text"].isdigit()), None)
        gill_regular, _ = _dominant_font(num_word, chars) if num_word else (gill_bold, font_size)

        header_words = header_line["words"] if header_line else words
        name_header_word = next((w for w in header_words if w["text"] == "Name"), None)
        if not name_header_word:
            name_header_word = next((w for w in words if w["text"] == "Name"), None)
        if not name_header_word:
            raise RuntimeError("Could not locate 'Name' header in template PDF.")
        georgia_bold, _ = _dominant_font(name_header_word, chars)

        fonts = TemplateFonts(
            gill_regular=gill_regular,
            gill_bold=gill_bold,
            georgia_bold=georgia_bold,
            size=font_size,
        )

        # Anchors
        anchors: Dict[str, float | List[float]] = {}
        anchors["grade_x"] = grade_word["x0"]
        anchors["grade_y"] = page_height - grade_word["bottom"]

        date_word = next(w for w in date_line["words"] if w["text"] == "Date")
        anchors["date_label_x"] = date_word["x0"]
        anchors["date_y"] = page_height - date_word["bottom"]
        day_word = next((w for w in date_line["words"] if w["text"].startswith("Day")), None)
        if day_word:
            after_day = [w for w in date_line["words"] if w["x0"] > day_word["x1"]]
            anchors["date_value_x"] = after_day[0]["x0"] if after_day else day_word["x1"] + 4
        else:
            anchors["date_value_x"] = date_word["x1"] + 4

        subject_word = next(w for w in subject_line["words"] if w["text"].startswith("Subject"))
        anchors["subject_label_x"] = subject_word["x0"]
        anchors["subject_y"] = page_height - subject_word["bottom"]
        after_subject = [w for w in subject_line["words"] if w["x0"] > subject_word["x1"]]
        anchors["subject_value_x"] = after_subject[0]["x0"] if after_subject else subject_word["x1"] + 4

        sl_word = next((w for w in header_words if w["text"].startswith("Sl")), None)
        if not sl_word:
            sl_word = next((w for w in words if w["text"].startswith("Sl")), None)
        if not sl_word:
            raise RuntimeError("Could not locate 'Sl.' header in template PDF.")
        anchors["slno_header_x"] = sl_word["x0"]
        anchors["slno_header_y"] = page_height - sl_word["bottom"]
        anchors["name_header_x"] = name_header_word["x0"]
        anchors["name_header_y"] = page_height - name_header_word["bottom"]

        present_word = next((w for w in header_words if w["text"].startswith("Present")), None)
        absent_word = next((w for w in header_words if w["text"].startswith("Absent")), None)
        if present_word:
            anchors["present_header_x"] = present_word["x0"]
            anchors["present_header_y"] = page_height - present_word["bottom"]
        if absent_word:
            anchors["absent_header_x"] = absent_word["x0"]
            anchors["absent_header_y"] = page_height - absent_word["bottom"]

        remarks_word = next((w for w in header_words if w["text"].startswith("Remarks")), None)
        if remarks_word:
            anchors["remarks_header_x"] = remarks_word["x0"]
            anchors["remarks_header_y"] = page_height - remarks_word["bottom"]

        # Row positions from numbers (restrict to Sl.No column and below header)
        number_words = [w for w in words if w["text"].isdigit()]
        header_y = page_height - sl_word["bottom"]
        footer_y = anchors.get("total_y", 0)
        row_numbers = []
        for w in number_words:
            y = page_height - w["bottom"]
            if y >= header_y - 1:
                continue
            if footer_y and y <= footer_y + 1:
                continue
            if w["x0"] > name_header_word["x0"] - 5:
                continue
            row_numbers.append(w)
        row_ys = sorted({page_height - w["bottom"] for w in row_numbers}, reverse=True)
        anchors["row_y"] = row_ys
        if row_numbers:
            anchors["slno_x"] = median(w["x0"] for w in row_numbers)
        else:
            anchors["slno_x"] = sl_word["x0"]
        anchors["name_x"] = name_header_word["x0"]

        # Footer lines (look below header row to avoid header text)
        header_top = 0
        if header_line:
            header_top = header_line.get("top") or (header_line.get("words", [{}])[0].get("top", 0))
        footer_lines = [l for l in grouped_lines if (l.get("top", 0) > header_top + 1)]
        footer_targets = {
            "total": ("Total", "No", "Students"),
            "present": ("No", "Students", "Present"),
            "absent": ("No", "Students", "Absent"),
            "invigilator": ("Invigilator",),
        }
        for key, needles in footer_targets.items():
            line = find_line(lambda l, n=needles: all(w in l["text"] for w in n), candidates=footer_lines)
            if not line:
                continue
            first_word = line["words"][0]
            anchors[f"{key}_x"] = first_word["x0"]
            anchors[f"{key}_y"] = page_height - first_word["bottom"]

        # Table bounds from vertical lines
        vertical_x = sorted({round(l.x0, 3) for l in lines_raw if abs(l.x0 - l.x1) < 0.01})
        if vertical_x:
            anchors["table_left_x"] = vertical_x[0]
            anchors["table_right_x"] = vertical_x[-1]
            anchors["vertical_x"] = vertical_x
            if len(vertical_x) >= 3:
                anchors["name_col_left_x"] = vertical_x[1]
                anchors["name_col_right_x"] = vertical_x[2]

        return TemplateLayout(
            page_width=page_width,
            page_height=page_height,
            lines=lines_raw,
            rects=rects_raw,
            fonts=fonts,
            anchors=anchors,
        )


In [3]:
from dataclasses import dataclass
from datetime import datetime, date
import re
from pathlib import Path
from typing import List, Tuple, Optional

import pdfplumber


@dataclass
class ExamSession:
    grade: int
    date: date
    day: str
    exam_slot: str
    subjects: List[str]
    times: List[str]


DATE_RE = re.compile(r'\b(\d{1,2})\s*([A-Za-z]{3,9})\b')
TIME_RANGE_RE = re.compile(
    r'(\d{1,2}[:.]\d{2}\s*(?:a\.?m\.?|p\.?m\.?))\s*[-–—?]\s*(\d{1,2}[:.]\d{2}\s*(?:a\.?m\.?|p\.?m\.?))',
    re.I,
)
TIME_TOKEN_RE = re.compile(r'\b\d{1,2}[:.]\d{2}\s*(?:a\.?m\.?|p\.?m\.?)\b', re.I)
GRADE_LINE_RE = re.compile(r'^\s*Grade\s*(\d{1,2})\s*$', re.I)
DATE_LINE_RE = re.compile(r'^\s*(\d{1,2})\s+([A-Za-z]{3,9})\s+(20\d{2})\s*\(([^)]+)\)\s*$', re.I)
EXAM_LINE_RE = re.compile(r'^\s*Exam\s*(\d+)\s*:\s*(.*)$', re.I)

NOISE_SUBJECT_RE = re.compile(
    r'\b(?:prep\s*time|exam\s*time|post\s*exam(?:\s*prep\s*time)?|after\s*exam|reading\s*time|no\s*exam)\b',
    re.I,
)


def _infer_year_from_filename(path: str | Path) -> Optional[int]:
    m = re.search(r'(20\d{2})', str(path))
    return int(m.group(1)) if m else None


def _parse_date_cell(cell: str, year: int) -> date:
    m = DATE_RE.search(cell)
    if not m:
        raise ValueError(f"Could not parse date from '{cell}'")
    day = int(m.group(1))
    mon = m.group(2)

    for fmt in ('%d %b %Y', '%d %B %Y'):
        try:
            dt = datetime.strptime(f"{day} {mon} {year}", fmt)
            return dt.date()
        except ValueError:
            continue

    raise ValueError(f"Could not parse date from '{cell}'")


def _normalize_spaces(text: str) -> str:
    return re.sub(r'\s+', ' ', text or '').strip()


def _strip_bracket_notes(text: str) -> str:
    def _drop_if_noise(match: re.Match) -> str:
        content = match.group(1)
        if re.search(r'(?:a\.?m\.?|p\.?m\.?|prep|exam\s*time|open\s*text|open\s*book|duration|mins?|hours?)', content, re.I):
            return ' '
        return match.group(0)

    text = re.sub(r'\(([^)]*)\)', _drop_if_noise, text)
    text = re.sub(r'\[([^]]*)\]', _drop_if_noise, text)
    return text


def _clean_subject(text: str) -> str:
    if not text:
        return ''

    text = _normalize_spaces(text)
    text = TIME_RANGE_RE.sub(' ', text)
    text = TIME_TOKEN_RE.sub(' ', text)
    text = _strip_bracket_notes(text)

    text = text.replace('|', ' ')
    text = re.sub(r'[\[\]{}"]', ' ', text)
    text = re.sub(r'\bopen\s*text\b', ' ', text, flags=re.I)
    text = re.sub(r'\bopen\s*book\b', ' ', text, flags=re.I)
    text = re.sub(r'\bexam\s*commence(?:ment)?\b', ' ', text, flags=re.I)

    if NOISE_SUBJECT_RE.search(text):
        text = NOISE_SUBJECT_RE.sub(' ', text)

    text = re.sub(r'\s{2,}', ' ', text)
    text = text.strip(' -,:;')
    return text


def _is_valid_subject(subject: str) -> bool:
    if not subject:
        return False

    low = subject.lower().strip()
    if low in {'exam time', 'prep time', 'no exam'}:
        return False
    if re.fullmatch(r'[-,:;\s]+', low):
        return False
    if not re.search(r'[a-z]', low):
        return False
    return True


def _split_subject_candidates(text: str) -> List[str]:
    text = _normalize_spaces(text)
    if not text:
        return []

    parts = [p.strip() for p in re.split(r'\s*[,;]\s*', text) if p.strip()]
    if not parts:
        return [text]

    merged: List[str] = []
    for part in parts:
        part = _normalize_spaces(part)
        if not part:
            continue

        if not merged:
            merged.append(part)
            continue

        low = part.lower()
        if re.match(r'^(paper|component|comp)\b', low):
            merged[-1] = f"{merged[-1]} {part}"
            continue

        if re.match(r'^\d+(?:\s*\([^)]+\))?$', low):
            merged[-1] = f"{merged[-1]} {part}"
            continue

        merged.append(part)

    return merged


def _parse_cell_text(cell: Optional[str]) -> List[Tuple[str, str]]:
    if not cell:
        return []

    lines = [ln.strip() for ln in cell.splitlines() if ln and ln.strip()]
    if not lines:
        return []

    text = _normalize_spaces(' '.join(lines))
    if not text or 'NO EXAM' in text.upper():
        return []

    candidates = _split_subject_candidates(text)
    if not candidates:
        candidates = [text]

    entries: List[Tuple[str, str]] = []
    seen = set()
    for candidate in candidates:
        time_match = TIME_RANGE_RE.search(candidate)
        time_val = normalize_time_range(time_match.group(0)) if time_match else ''

        subject = _clean_subject(candidate)
        if not _is_valid_subject(subject):
            continue

        key = subject.lower()
        if key in seen:
            continue
        seen.add(key)
        entries.append((subject, time_val))

    return entries


def _parse_schedule_from_tables(pdf_path: Path, year: int) -> List[ExamSession]:
    sessions: List[ExamSession] = []
    current_date: Optional[date] = None
    current_day: str = ''

    with pdfplumber.open(str(pdf_path)) as pdf:
        for page in pdf.pages:
            tables = page.extract_tables() or []
            for table in tables:
                for row in table:
                    if not row or len(row) < 7:
                        continue
                    if row[0] and str(row[0]).strip().lower() == 'date':
                        continue

                    date_cell = (row[0] or '').strip() if row[0] else ''
                    day_cell = (row[1] or '').strip() if row[1] else ''
                    exam_slot = (row[2] or '').strip() if row[2] else ''

                    if date_cell:
                        current_date = _parse_date_cell(date_cell, year)
                    if day_cell:
                        current_day = day_cell
                    if not current_date:
                        continue

                    for offset, grade in enumerate([9, 10, 11, 12]):
                        cell_text = row[3 + offset] if len(row) > 3 + offset else ''
                        entries = _parse_cell_text(cell_text)
                        if not entries:
                            continue

                        sessions.append(
                            ExamSession(
                                grade=grade,
                                date=current_date,
                                day=current_day or current_date.strftime('%A'),
                                exam_slot=exam_slot or '',
                                subjects=[e[0] for e in entries],
                                times=[e[1] for e in entries],
                            )
                        )

    return sessions


def _parse_schedule_from_text(pdf_path: Path, default_year: int) -> List[ExamSession]:
    sessions: List[ExamSession] = []

    current_grade: Optional[int] = None
    current_date: Optional[date] = None
    current_day: str = ''

    pending_slot: str = ''
    pending_body: str = ''

    def flush_pending() -> None:
        nonlocal pending_slot, pending_body

        if not (current_grade and current_date and pending_slot):
            pending_slot = ''
            pending_body = ''
            return

        entries = _parse_cell_text(pending_body)
        if entries:
            sessions.append(
                ExamSession(
                    grade=current_grade,
                    date=current_date,
                    day=current_day or current_date.strftime('%A'),
                    exam_slot=pending_slot,
                    subjects=[e[0] for e in entries],
                    times=[e[1] for e in entries],
                )
            )

        pending_slot = ''
        pending_body = ''

    with pdfplumber.open(str(pdf_path)) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text() or ''
            lines = [ln.strip() for ln in page_text.splitlines() if ln and ln.strip()]

            for line in lines:
                line = _normalize_spaces(line)
                if not line:
                    continue

                grade_match = GRADE_LINE_RE.match(line)
                if grade_match:
                    flush_pending()
                    current_grade = int(grade_match.group(1))
                    continue

                date_match = DATE_LINE_RE.match(line)
                if date_match:
                    flush_pending()

                    day_num = int(date_match.group(1))
                    month_text = date_match.group(2)
                    year_text = int(date_match.group(3)) if date_match.group(3) else default_year

                    parsed_date = None
                    for fmt in ('%d %b %Y', '%d %B %Y'):
                        try:
                            parsed_date = datetime.strptime(f"{day_num} {month_text} {year_text}", fmt).date()
                            break
                        except ValueError:
                            continue

                    if parsed_date:
                        current_date = parsed_date
                        current_day = date_match.group(4).strip()
                    continue

                exam_match = EXAM_LINE_RE.match(line)
                if exam_match:
                    flush_pending()
                    pending_slot = f"EXAM {int(exam_match.group(1))}"
                    pending_body = exam_match.group(2).strip()
                    continue

                if pending_slot:
                    pending_body = _normalize_spaces(f"{pending_body} {line}")

        flush_pending()

    return sessions


def parse_schedule(pdf_path: str | Path, year: Optional[int] = None) -> List[ExamSession]:
    pdf_path = Path(pdf_path)
    year = year or _infer_year_from_filename(pdf_path) or datetime.now().year

    sessions = _parse_schedule_from_tables(pdf_path, year)
    if sessions:
        return sessions

    return _parse_schedule_from_text(pdf_path, year)


In [4]:
import re
from pathlib import Path
from typing import Dict, List
import pdfplumber
from collections import defaultdict

GRADE_SECTION_RE = re.compile(r'Grade\s*(\d{1,2})\s*([A-Za-z])')
SECTION_TOKEN_RE = re.compile(r'^(\d{1,2})([A-Za-z])$')


def _group_words_by_line(words, tolerance: float = 1.5):
    lines = []
    for w in sorted(words, key=lambda x: x['top']):
        placed = False
        for line in lines:
            if abs(line['top'] - w['top']) <= tolerance:
                line['words'].append(w)
                placed = True
                break
        if not placed:
            lines.append({'top': w['top'], 'words': [w]})
    for line in lines:
        line['words'].sort(key=lambda x: x['x0'])
        line['text'] = ' '.join(w['text'] for w in line['words']).strip()
    return lines


def _find_column_headers(lines):
    headers = []
    for line in lines:
        words = line['words']
        for i, w in enumerate(words[:-1]):
            if w['text'].lower() == 'grade':
                nxt = words[i + 1]['text']
                m = SECTION_TOKEN_RE.match(nxt)
                if m:
                    grade = int(m.group(1))
                    section = m.group(2).upper()
                    headers.append((grade, section, words[i + 1]))
    return headers


def _columns_from_headers(headers, page_width):
    cols = []
    centers = []
    for grade, section, w in headers:
        center = (w['x0'] + w['x1']) / 2
        centers.append((center, grade, section))
    centers.sort(key=lambda x: x[0])
    if not centers:
        return []
    bounds = [0.0]
    for idx in range(len(centers) - 1):
        mid = (centers[idx][0] + centers[idx + 1][0]) / 2
        bounds.append(mid)
    bounds.append(page_width + 1)

    for idx, (center, grade, section) in enumerate(centers):
        left = bounds[idx]
        right = bounds[idx + 1]
        cols.append((grade, section, left, right))
    return cols


def _clean_name_fragment(text: str) -> str:
    text = re.sub(r'\b\d+\b', ' ', text)
    text = re.sub(r'[_|]+', ' ', text)
    text = re.sub(r'\s{2,}', ' ', text)
    return text.strip(' ,;:-')


def _extract_names_from_line(raw_line: str) -> List[str]:
    if not raw_line:
        return []

    candidates = []
    for part in split_multi_name_line(raw_line):
        part = part.strip()
        if not part:
            continue

        # In some PDFs, roll numbers are embedded between names (e.g., "Arjun 1 Sachin 4").
        split_by_digits = [p.strip() for p in re.split(r'\b\d+\b', part) if p.strip()]
        if not split_by_digits:
            split_by_digits = [part]

        for chunk in split_by_digits:
            cleaned = _clean_name_fragment(chunk)
            if cleaned and re.search(r'[A-Za-z]', cleaned):
                candidates.append(cleaned)

    seen = set()
    result: List[str] = []
    for name in candidates:
        key = name.lower()
        if key in seen:
            continue
        seen.add(key)
        result.append(name)
    return result


def parse_students(pdf_path: str | Path) -> Dict[int, Dict[str, List[str]]]:
    pdf_path = Path(pdf_path)
    data: Dict[int, Dict[str, List[str]]] = defaultdict(lambda: defaultdict(list))

    with pdfplumber.open(str(pdf_path)) as pdf:
        for page in pdf.pages:
            words = page.extract_words()
            if not words:
                continue

            lines = _group_words_by_line(words)
            headers = _find_column_headers(lines)

            if headers:
                columns = _columns_from_headers(headers, page.width)
                for grade, section, left, right in columns:
                    col_words = [w for w in words if w['x0'] >= left and w['x0'] < right]
                    digit_words = [w for w in col_words if w['text'].isdigit()]
                    if not digit_words:
                        continue
                    start_top = min(w['top'] for w in digit_words) - 0.5
                    content_words = [w for w in col_words if w['top'] >= start_top]
                    col_lines = _group_words_by_line(content_words)
                    for line in col_lines:
                        tokens = [w['text'] for w in line['words']]
                        if not tokens:
                            continue
                        if any('@' in t for t in tokens):
                            continue
                        if tokens[0].lower().startswith('sl'):
                            continue
                        if tokens[0].isdigit():
                            tokens = tokens[1:]
                        if not tokens:
                            continue

                        raw_name = ' '.join(tokens).strip()
                        for cleaned_name in _extract_names_from_line(raw_name):
                            data[grade][section].append(cleaned_name)
                continue

            # Fallback: sequential list by grade/section headers
            current_grade = None
            current_section = None
            skip_tokens = {
                'sl.',
                'no',
                'name',
                'present/absent',
                'present/',
                'absent',
                'remarks',
                'date & day:',
                'subject:',
            }
            text = page.extract_text() or ''
            for raw_line in text.splitlines():
                line = raw_line.strip()
                if not line:
                    continue
                m = GRADE_SECTION_RE.search(line)
                if m:
                    current_grade = int(m.group(1))
                    current_section = m.group(2).upper()
                    continue
                low = line.lower()
                if any(tok in low for tok in [
                    'total no of students',
                    'no of students present',
                    'no of students absent',
                    'name of the invigilator',
                ]):
                    continue
                if low in skip_tokens or low.startswith('sl.') or 'sl. no' in low:
                    continue
                if 'date & day' in low or low.startswith('subject:'):
                    continue
                if current_grade is None or current_section is None:
                    continue

                line = re.sub(r'^\d+\s+', '', line).strip()
                if not line:
                    continue

                for cleaned_name in _extract_names_from_line(line):
                    data[current_grade][current_section].append(cleaned_name)

    return data


In [5]:
from dataclasses import dataclass
from pathlib import Path
from statistics import median
from typing import Dict, List, Optional, Tuple
from reportlab.pdfgen import canvas
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont


@dataclass
class FontMapping:
    gill_regular: str
    gill_bold: str
    georgia_bold: str


def _usable_ttf(path: Path, min_size: int = 60000) -> bool:
    return path.exists() and path.stat().st_size >= min_size


def _register_ttf(path: Path, alias: str) -> str:
    if alias not in pdfmetrics.getRegisteredFontNames():
        pdfmetrics.registerFont(TTFont(alias, str(path)))
    return alias


def register_fonts(font_dir: str | Path, template_fonts: TemplateFonts) -> FontMapping:
    font_dir = Path(font_dir)
    font_dir.mkdir(parents=True, exist_ok=True)
    system_fonts = Path(r'C:\Windows\Fonts')

    mapping = {}
    fallbacks = {
        'gill_regular': 'Helvetica',
        'gill_bold': 'Helvetica-Bold',
        'georgia_bold': 'Times-Bold',
    }

    candidates = {
        'gill_regular': [
            system_fonts / 'GillSansMT.ttf',
            system_fonts / 'arial.ttf',
            font_dir / f"{template_fonts.gill_regular}.ttf",
        ],
        'gill_bold': [
            system_fonts / 'GillSansMT-Bold.ttf',
            system_fonts / 'arialbd.ttf',
            font_dir / f"{template_fonts.gill_bold}.ttf",
        ],
        'georgia_bold': [
            system_fonts / 'georgiab.ttf',
            font_dir / f"{template_fonts.georgia_bold}.ttf",
        ],
    }

    for key, paths in candidates.items():
        chosen = None
        for path in paths:
            if _usable_ttf(path):
                chosen = path
                break
        if chosen:
            alias = f"{key}_font"
            mapping[key] = _register_ttf(chosen, alias)
        else:
            mapping[key] = fallbacks[key]

    return FontMapping(**mapping)


def _fit_font_size(text: str, font_name: str, base_size: float, max_width: float, min_size: float = 8.0) -> float:
    size = float(base_size)
    while size > min_size and pdfmetrics.stringWidth(text, font_name, size) > max_width:
        size -= 0.25
    return max(size, min_size)


def _truncate_to_width(text: str, font_name: str, font_size: float, max_width: float) -> str:
    if pdfmetrics.stringWidth(text, font_name, font_size) <= max_width:
        return text

    ellipsis = '...'
    trimmed = text
    while trimmed and pdfmetrics.stringWidth(trimmed + ellipsis, font_name, font_size) > max_width:
        trimmed = trimmed[:-1]
    return (trimmed.rstrip() + ellipsis) if trimmed else ellipsis


def _resolve_columns(layout: TemplateLayout) -> Tuple[float, float, float, float, float, List[float], float, float, float, float]:
    vertical_x = layout.anchors.get('vertical_x')
    all_verticals = [l for l in layout.lines if abs(l.x0 - l.x1) < 0.01]
    if all_verticals:
        table_bottom = min(min(l.y0, l.y1) for l in all_verticals)
        table_top = max(max(l.y0, l.y1) for l in all_verticals)
    else:
        table_bottom = 0.0
        table_top = layout.page_height

    if not vertical_x or len(vertical_x) < 5:
        slno_left = layout.anchors.get('table_left_x', 50.0)
        slno_right = layout.anchors.get('name_col_left_x', slno_left + 48.0)
        name_left = slno_right
        name_right = layout.anchors.get('name_col_right_x', name_left + 200.0)
        present_right = layout.anchors.get('table_right_x', name_right + 160.0)
        return (
            float(slno_left),
            float(slno_right),
            float(name_left),
            float(name_right),
            float(present_right),
            [],
            float(table_bottom),
            float(table_top),
            0.7,
            float(present_right),
        )

    left, old_slno, old_name, old_present, right = [float(x) for x in vertical_x[:5]]

    slno_right = max(left + 42.0, old_slno - 10.0)
    name_right = old_name - 8.0

    # Widen Present/Absent & Remarks towards the right without shrinking Name.
    remarks_expand = 18.0
    right_limit = float(layout.page_width - 36.0)
    present_right = min(right + remarks_expand, right_limit)

    min_name_width = 170.0
    min_present_width = 145.0
    if name_right - slno_right < min_name_width:
        name_right = slno_right + min_name_width
    if right - name_right < min_present_width:
        name_right = right - min_present_width

    name_right = max(name_right, slno_right + 120.0)
    name_right = min(name_right, right - 110.0)

    interior_lines = [
        l
        for l in all_verticals
        if any(abs(l.x0 - x) < 0.6 for x in (old_slno, old_name, old_present))
    ]
    if interior_lines:
        table_bottom = min(min(l.y0, l.y1) for l in interior_lines)
        table_top = max(max(l.y0, l.y1) for l in interior_lines)
        line_width = median([l.width for l in interior_lines if l.width]) if interior_lines else 0.7
    else:
        line_width = 0.7

    skip_verticals = [old_slno, old_name, old_present]
    if present_right > right + 0.1:
        skip_verticals.append(right)

    return (
        left,
        slno_right,
        slno_right,
        name_right,
        present_right,
        skip_verticals,
        table_bottom,
        table_top,
        float(line_width),
        right,
    )


def _body_row_bounds(layout: TemplateLayout, base_rows: List[float]) -> Tuple[Optional[float], Optional[float], List[float]]:
    if not base_rows:
        return None, None, []

    h_lines = sorted({float(l.y0) for l in layout.lines if abs(l.y0 - l.y1) < 0.01}, reverse=True)
    if not h_lines:
        return None, None, []

    first_baseline = float(base_rows[0])
    last_baseline = float(base_rows[-1])

    # Student body starts at the line directly below the column headers.
    header_y = float(layout.anchors.get('slno_header_y', first_baseline + 18.0))
    top = max((y for y in h_lines if y < header_y - 0.1), default=None)

    footer_total_y = layout.anchors.get('total_y')
    if footer_total_y is not None:
        footer_ref = float(footer_total_y) + 6.0
        bottom_candidates = [y for y in h_lines if y > footer_ref and (top is None or y < top - 0.1)]
        bottom = min(bottom_candidates) if bottom_candidates else None
    else:
        bottom = None

    if top is None:
        top = max((y for y in h_lines if y < first_baseline + 20.0), default=None)
    if bottom is None:
        bottom = max((y for y in h_lines if y < last_baseline + 0.8), default=None)

    if top is None:
        top = first_baseline + 14.0
    if bottom is None:
        bottom = last_baseline - 2.5

    if top <= bottom:
        return None, None, []

    body_h = [y for y in h_lines if bottom - 0.2 <= y <= top + 0.2]
    return top, bottom, body_h


def _compute_student_positions(
    base_rows: List[float],
    student_count: int,
    base_font_size: float,
    body_top: Optional[float] = None,
    body_bottom: Optional[float] = None,
) -> Tuple[List[float], float]:
    if not base_rows or student_count <= 0:
        return [], base_font_size

    if student_count <= len(base_rows):
        return base_rows[:student_count], base_font_size

    if body_top is None or body_bottom is None or body_top <= body_bottom:
        top_y = float(base_rows[0]) + 14.0
        bottom_y = float(base_rows[-1])
    else:
        top_y = float(body_top)
        bottom_y = float(body_bottom)

    row_height = (top_y - bottom_y) / float(student_count)

    # Keep baseline safely inside each compact row.
    baseline_offset = row_height * 0.68
    baseline_offset = max(2.0, min(baseline_offset, row_height - 0.9))
    row_positions = [top_y - i * row_height - baseline_offset for i in range(student_count)]

    row_font = max(6.6, min(base_font_size - 1.5, row_height - 2.2))
    return row_positions, row_font


def draw_attendance_page(
    c: canvas.Canvas,
    layout: TemplateLayout,
    fonts: FontMapping,
    grade: int,
    section: str,
    date_str: str,
    subject_text: str,
    students: List[str],
    include_time_in_subject: bool = False,
    time_texts: Optional[List[str]] = None,
    row_start: int = 1,
):
    base_rows = layout.anchors.get('row_y', [])
    body_top, body_bottom, body_h_lines = _body_row_bounds(layout, base_rows)
    compress_rows = (
        len(base_rows) > 0
        and len(students) > len(base_rows)
        and body_top is not None
        and body_bottom is not None
        and body_top > body_bottom
    )

    for rect in layout.rects:
        r, g, b = rect.fill_rgb

        # Keep footer cells unfilled (no green highlight).
        if r <= 0.2 and g >= 0.8 and b <= 0.2:
            continue

        c.setFillColorRGB(r, g, b)
        c.rect(rect.x0, rect.y0, rect.x1 - rect.x0, rect.y1 - rect.y0, stroke=0, fill=1)

    (
        slno_left,
        slno_right,
        name_left,
        name_right,
        present_right,
        skip_vertical_x,
        table_bottom,
        table_top,
        custom_line_width,
        old_right,
    ) = _resolve_columns(layout)
    present_left = name_right

    # Extend the beige header background to the widened right edge.
    if present_right > old_right + 0.1:
        for rect in layout.rects:
            r, g, b = rect.fill_rgb
            if not (r >= 0.90 and g >= 0.85 and b >= 0.70):
                continue
            if abs(rect.x1 - old_right) > 1.0:
                continue
            c.setFillColorRGB(r, g, b)
            c.rect(old_right, rect.y0, present_right - old_right, rect.y1 - rect.y0, stroke=0, fill=1)

    c.setStrokeColorRGB(0, 0, 0)
    for line in layout.lines:
        if abs(line.x0 - line.x1) < 0.01 and any(abs(line.x0 - x) < 0.6 for x in skip_vertical_x):
            continue

        if compress_rows and abs(line.y0 - line.y1) < 0.01:
            y = float(line.y0)
            if body_bottom - 0.2 <= y <= body_top + 0.2:
                # Replace original row grid with compressed grid for high-count sections.
                continue

        lw = line.width if line.width and line.width > 0 else 0.5
        c.setLineWidth(lw)
        c.line(line.x0, line.y0, line.x1, line.y1)

    if skip_vertical_x:
        c.setLineWidth(custom_line_width if custom_line_width > 0 else 0.7)
        c.line(slno_right, table_bottom, slno_right, table_top)
        c.line(name_right, table_bottom, name_right, table_top)
        if present_right > old_right + 0.1:
            c.line(present_right, table_bottom, present_right, table_top)

    if present_right > old_right + 0.1:
        header_top = table_top
        for line in layout.lines:
            if abs(line.y0 - line.y1) >= 0.01:
                continue

            y = float(line.y0)
            x_right = max(float(line.x0), float(line.x1))
            if abs(x_right - old_right) > 1.0:
                continue

            # Top beige header rows sit above table_top; extend those borders too.
            if y > table_top + 0.2:
                lw = line.width if line.width and line.width > 0 else 0.5
                c.setLineWidth(lw)
                c.line(old_right, y, present_right, y)
                header_top = max(header_top, y)
                continue

            if not (table_bottom - 0.2 <= y <= table_top + 0.2):
                continue

            if compress_rows and body_top is not None and body_bottom is not None and body_bottom - 0.2 <= y <= body_top + 0.2:
                continue

            lw = line.width if line.width and line.width > 0 else 0.5
            c.setLineWidth(lw)
            c.line(old_right, y, present_right, y)

        # Close the right border through the top beige header area.
        if header_top > table_top + 0.2:
            c.setLineWidth(custom_line_width if custom_line_width > 0 else 0.7)
            c.line(present_right, table_top, present_right, header_top)

    if compress_rows:
        row_count = len(students)
        row_height = (body_top - body_bottom) / float(row_count)
        c.setLineWidth(custom_line_width if custom_line_width > 0 else 0.7)
        for i in range(row_count + 1):
            y = body_top - i * row_height
            c.line(slno_left, y, present_right, y)

    c.setFillColorRGB(0, 0, 0)
    size = float(layout.fonts.size)

    c.setFont(fonts.gill_bold, size)
    c.drawString(layout.anchors['grade_x'], layout.anchors['grade_y'], f"Grade {grade}{section}")

    c.drawString(layout.anchors['date_label_x'], layout.anchors['date_y'], 'Date & Day:')
    c.drawString(layout.anchors['date_value_x'], layout.anchors['date_y'], date_str)

    c.drawString(layout.anchors['subject_label_x'], layout.anchors['subject_y'], 'Subject:')
    subject_value_x = layout.anchors['subject_value_x']
    right_x = layout.anchors.get('table_right_x', layout.page_width - 40)
    subject_max_width = right_x - subject_value_x - 2

    final_subject = subject_text
    if include_time_in_subject and time_texts:
        time_bits = [t for t in time_texts if t]
        if time_bits:
            final_subject = f"{subject_text} ({'; '.join(time_bits)})"

    subject_size = _fit_font_size(final_subject, fonts.gill_bold, size, subject_max_width, min_size=8.0)
    subject_draw = _truncate_to_width(final_subject, fonts.gill_bold, subject_size, subject_max_width)
    c.setFont(fonts.gill_bold, subject_size)
    c.drawString(subject_value_x, layout.anchors['subject_y'] + (size - subject_size) * 0.1, subject_draw)

    c.setFont(fonts.gill_bold, size)
    c.drawString(slno_left + 6, layout.anchors['slno_header_y'], 'Sl. No')

    name_text = 'Name'
    name_center = (name_left + name_right) / 2
    name_x = name_center - pdfmetrics.stringWidth(name_text, fonts.georgia_bold, size) / 2
    c.setFont(fonts.georgia_bold, size)
    c.drawString(name_x, layout.anchors['name_header_y'], name_text)

    pa_text = 'Present/Absent & Remarks'
    pa_max_width = max(20, (present_right - present_left) - 8)
    pa_size = _fit_font_size(pa_text, fonts.gill_bold, size, pa_max_width, min_size=8.0)
    pa_center = (present_left + present_right) / 2
    pa_x = pa_center - pdfmetrics.stringWidth(pa_text, fonts.gill_bold, pa_size) / 2
    c.setFont(fonts.gill_bold, pa_size)
    c.drawString(pa_x, layout.anchors['slno_header_y'] + (size - pa_size) * 0.1, pa_text)

    row_positions, row_font = _compute_student_positions(
        base_rows,
        len(students),
        size,
        body_top=body_top,
        body_bottom=body_bottom,
    )
    slno_x_text = slno_left + 6
    name_x_text = name_left + 4
    max_name_width = max(10, name_right - name_x_text - 4)

    for idx, row_y in enumerate(row_positions):
        c.setFont(fonts.gill_regular, row_font)
        c.drawString(slno_x_text, row_y, str(row_start + idx))

        name = students[idx]
        name_size = _fit_font_size(name, fonts.gill_regular, row_font, max_name_width, min_size=max(6.4, row_font - 1.5))
        draw_name = _truncate_to_width(name, fonts.gill_regular, name_size, max_name_width)
        c.setFont(fonts.gill_regular, name_size)
        c.drawString(name_x_text, row_y + (row_font - name_size) * 0.2, draw_name)

    c.setFont(fonts.gill_regular, size)
    if 'total_x' in layout.anchors:
        c.drawString(layout.anchors['total_x'], layout.anchors['total_y'], 'Total No of Students')
    if 'present_x' in layout.anchors:
        c.drawString(layout.anchors['present_x'], layout.anchors['present_y'], 'No of Students Present')
    if 'absent_x' in layout.anchors:
        c.drawString(layout.anchors['absent_x'], layout.anchors['absent_y'], 'No of Students Absent')
    if 'invigilator_x' in layout.anchors:
        c.drawString(layout.anchors['invigilator_x'], layout.anchors['invigilator_y'], 'Name of the Invigilator')


def render_grade_pdf(
    output_path: str | Path,
    layout: TemplateLayout,
    fonts: FontMapping,
    grade: int,
    sessions: List[Tuple[str, str, List[str]]],
    students_by_section: Dict[str, List[str]],
    include_time_in_subject: bool = False,
):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    c = canvas.Canvas(str(output_path), pagesize=(layout.page_width, layout.page_height))
    max_students_per_page = 30

    for session in sessions:
        date_str, subject_text, time_texts = session
        for section in sorted(students_by_section.keys()):
            students = students_by_section[section]
            if len(students) <= max_students_per_page:
                chunks = [(0, students)]
            else:
                chunks = [(start, students[start : start + max_students_per_page]) for start in range(0, len(students), max_students_per_page)]

            for start, chunk in chunks:
                draw_attendance_page(
                    c,
                    layout,
                    fonts,
                    grade,
                    section,
                    date_str,
                    subject_text,
                    chunk,
                    include_time_in_subject=include_time_in_subject,
                    time_texts=time_texts,
                    row_start=start + 1,
                )
                c.showPage()

    c.save()








In [6]:
from collections import OrderedDict
from pathlib import Path
import re
from typing import Optional, List, Dict, Tuple, Union

StudentsInput = Union[str, Path, Dict[int, Union[str, Path]], List[Union[str, Path]], Tuple[Union[str, Path], ...]]

_TIME_RANGE_OR_TOKEN_RE = re.compile(
    r'\b\d{1,2}[:.]\d{2}\s*(?:a\.?m\.?|p\.?m\.?)\b(?:\s*[-–—?]\s*\d{1,2}[:.]\d{2}\s*(?:a\.?m\.?|p\.?m\.?)\b)?',
    re.I,
)
_PAPER_COMPONENT_RE = re.compile(
    r'^(?P<subject>.+?)\s+(?P<kind>Paper|Component|Comp)\s+(?P<part>[0-9A-Za-z]+(?:\s*/\s*[0-9A-Za-z]+)?(?:\s*\([^)]+\))?)$',
    re.I,
)


def _merge_students(dest: Dict[int, Dict[str, List[str]]], src: Dict[int, Dict[str, List[str]]]) -> None:
    for g, sections in src.items():
        if g not in dest:
            dest[g] = {}
        for sec, names in sections.items():
            dest[g][sec] = names


def _load_students_data(students_pdf: StudentsInput) -> Dict[int, Dict[str, List[str]]]:
    merged: Dict[int, Dict[str, List[str]]] = {}

    if isinstance(students_pdf, (str, Path)):
        p = Path(students_pdf)
        if not p.exists():
            raise FileNotFoundError(f'Student list PDF not found: {p}')
        _merge_students(merged, parse_students(p))
        return merged

    if isinstance(students_pdf, dict):
        for grade_key, path_value in students_pdf.items():
            p = Path(path_value)
            if not p.exists():
                raise FileNotFoundError(f'Student list PDF not found for Grade {grade_key}: {p}')

            parsed = parse_students(p)
            expected_grade = int(grade_key)

            if expected_grade in parsed:
                _merge_students(merged, {expected_grade: parsed[expected_grade]})
                continue

            if len(parsed) == 1:
                only_grade = next(iter(parsed))
                _merge_students(merged, {expected_grade: parsed[only_grade]})
            else:
                raise ValueError(
                    f'Could not map student file {p} to Grade {expected_grade}. '
                    f'Grades found in file: {sorted(parsed.keys())}'
                )
        return merged

    if isinstance(students_pdf, (list, tuple)):
        for path_value in students_pdf:
            p = Path(path_value)
            if not p.exists():
                raise FileNotFoundError(f'Student list PDF not found: {p}')
            _merge_students(merged, parse_students(p))
        return merged

    raise TypeError('students_pdf must be a path, dict of grade->path, or list of paths.')


def _join_with_and(parts: List[str]) -> str:
    if not parts:
        return ''
    if len(parts) == 1:
        return parts[0]
    if len(parts) == 2:
        return f'{parts[0]} and {parts[1]}'
    return ', '.join(parts[:-1]) + f' and {parts[-1]}'


def _clean_exam_title(raw: str) -> str:
    text = ' '.join((raw or '').split())
    if not text:
        return ''

    text = _TIME_RANGE_OR_TOKEN_RE.sub('', text)
    text = re.sub(r'[\[\]{}"]', ' ', text)
    text = re.sub(r'\b(?:prep\s*time|exam\s*time|post\s*exam(?:\s*prep\s*time)?|after\s*exam|reading\s*time|open\s*text|open\s*book|no\s*exam)\b', ' ', text, flags=re.I)
    text = re.sub(r'\((?:[^)]*(?:a\.?m\.?|p\.?m\.?|prep|exam\s*time|open\s*text|open\s*book|duration|mins?|hours?)[^)]*)\)', ' ', text, flags=re.I)
    text = re.sub(r'\[(?:[^]]*(?:a\.?m\.?|p\.?m\.?|prep|exam\s*time|open\s*text|open\s*book|duration|mins?|hours?)[^]]*)\]', ' ', text, flags=re.I)
    text = text.replace('|', ' ')
    text = re.sub(r'\s+,', ',', text)
    text = re.sub(r'\s{2,}', ' ', text)
    text = text.strip(' -,:;')

    if text.lower() in {'', 'exam time', 'prep time', 'no exam'}:
        return ''
    if re.fullmatch(r'[-,:;\s]+', text):
        return ''
    return text


def _format_session_subjects(subjects: List[str]) -> str:
    cleaned: List[str] = []
    seen = set()
    for subject in subjects:
        title = _clean_exam_title(subject)
        if not title:
            continue
        key = title.lower()
        if key in seen:
            continue
        seen.add(key)
        cleaned.append(title)

    grouped: Dict[Tuple[str, str], Dict[str, object]] = {}
    order: List[Tuple[str, object]] = []

    for title in cleaned:
        m = _PAPER_COMPONENT_RE.match(title)
        if m:
            part = m.group('part').strip()
            if re.search(r'\bday\s*\d+\b', part, re.I):
                order.append(('single', title))
                continue

            subject_name = ' '.join(m.group('subject').split())
            kind = 'Paper' if m.group('kind').lower().startswith('paper') else 'Component'
            key = (subject_name.lower(), kind.lower())

            if key not in grouped:
                grouped[key] = {'subject': subject_name, 'kind': kind, 'parts': []}
                order.append(('group', key))

            parts = grouped[key]['parts']
            if part not in parts:
                parts.append(part)
        else:
            order.append(('single', title))

    combined: List[str] = []
    out_seen = set()
    for tag, value in order:
        if tag == 'single':
            line = str(value)
        else:
            group = grouped[value]
            parts = group['parts']
            if len(parts) == 1:
                line = f"{group['subject']} {group['kind']} {parts[0]}"
            else:
                line = f"{group['subject']} {group['kind']} {_join_with_and(parts)}"

        line = re.sub(r'\s{2,}', ' ', line).strip(' -,:;')
        if not line:
            continue

        key = line.lower()
        if key in out_seen:
            continue
        out_seen.add(key)
        combined.append(line)

    return ', '.join(combined)


def generate_attendance_pdfs(
    schedule_pdf: str | Path,
    students_pdf: StudentsInput,
    template_pdf: str | Path,
    output_dir: str | Path = 'output',
    grade: Optional[int] = None,
    grades: Optional[List[int]] = None,
    year: Optional[int] = None,
    include_times: bool = False,
):
    if not schedule_pdf or not students_pdf:
        raise ValueError('Both schedule PDF and student list PDF are required for generation.')

    schedule_pdf = Path(schedule_pdf)
    template_pdf = Path(template_pdf)
    output_dir = Path(output_dir)

    if not schedule_pdf.exists():
        raise FileNotFoundError(f'Schedule PDF not found: {schedule_pdf}')
    if not template_pdf.exists():
        raise FileNotFoundError(f'Template PDF not found: {template_pdf}')

    selected_grades = None
    if grades:
        selected_grades = {int(g) for g in grades}
    elif grade is not None:
        selected_grades = {int(grade)}

    font_dir = output_dir / '_fonts'
    extract_fonts_from_template(template_pdf, font_dir)
    layout = analyze_template(template_pdf)
    fonts = register_fonts(font_dir, layout.fonts)

    sessions = parse_schedule(schedule_pdf, year=year)
    students = _load_students_data(students_pdf)

    by_grade: Dict[int, List[ExamSession]] = {}
    for s in sessions:
        by_grade.setdefault(s.grade, []).append(s)

    slot_order = {'EXAM 1': 1, 'EXAM 2': 2, 'EXAM 3': 3}
    for g in sorted(by_grade.keys()):
        if selected_grades and g not in selected_grades:
            continue
        if g not in students:
            print(f'Warning: No students found for Grade {g}. Skipping.')
            continue

        grouped_sessions: OrderedDict[Tuple[object, str, str], Dict[str, List[str]]] = OrderedDict()
        for s in by_grade[g]:
            slot = (s.exam_slot or '').strip().upper()
            key = (s.date, s.day, slot)
            if key not in grouped_sessions:
                grouped_sessions[key] = {'subjects': [], 'times': []}
            grouped_sessions[key]['subjects'].extend(s.subjects)
            grouped_sessions[key]['times'].extend([t for t in s.times if t])

        sorted_keys = sorted(
            grouped_sessions.keys(),
            key=lambda key: (key[0], slot_order.get(key[2], 99), key[2]),
        )

        session_tuples: List[Tuple[str, str, List[str]]] = []
        for key in sorted_keys:
            session_info = grouped_sessions[key]
            subject_text = _format_session_subjects(session_info['subjects'])
            if not subject_text:
                continue

            date_obj, day_text, _slot = key
            date_str = format_date_with_day(date_obj, day_text)
            time_texts = session_info['times'] if include_times else []
            session_tuples.append((date_str, subject_text, time_texts))

        output_path = output_dir / f'Grade_{g}_Attendance.pdf'
        render_grade_pdf(
            output_path=output_path,
            layout=layout,
            fonts=fonts,
            grade=g,
            sessions=session_tuples,
            students_by_section=students[g],
            include_time_in_subject=include_times,
        )
        print(f'Generated {output_path}')


## Example usage (do not run yet)
Fill in the three PDF paths below, then uncomment the call.


In [7]:
import re
from pathlib import Path
from typing import Dict, List

import pdfplumber

TIME_ARTIFACT_RE = re.compile(r'\b\d{1,2}[:.]\d{2}\s*(?:a\.?m\.?|p\.?m\.?)\b', re.I)
STUDENT_LINE_RE = re.compile(r'^\s*(\d+)\s+(.+?)\s*$')
NAME_DIGIT_RE = re.compile(r'\b\d+\b')


def _detect_empty_first_body_row(page) -> Dict[str, float] | None:
    words = page.extract_words() or []
    if not words:
        return None

    h_lines = sorted({float(l['y0']) for l in page.lines if abs(l['y0'] - l['y1']) < 0.01}, reverse=True)
    if len(h_lines) < 3:
        return None

    header_words = [w for w in words if w['text'].lower().startswith('sl')]
    if not header_words:
        return None

    header_y = max(page.height - w['bottom'] for w in header_words)
    body_lines = [y for y in h_lines if y < header_y - 0.1]
    if len(body_lines) < 2:
        return None

    body_top = body_lines[0]
    body_second = body_lines[1]

    vertical_x = sorted({float(l['x0']) for l in page.lines if abs(l['x0'] - l['x1']) < 0.01})
    slno_right = vertical_x[1] if len(vertical_x) >= 2 else 130.0

    one_row_y = []
    for w in words:
        if w['text'] != '1':
            continue
        if w['x0'] > slno_right - 2:
            continue

        y = page.height - w['bottom']
        if y <= body_top + 2:
            one_row_y.append(y)

    if not one_row_y:
        return None

    y1 = max(one_row_y)
    if body_second < y1 < body_top:
        return None

    return {'row1_y': y1, 'top_line': body_top, 'second_line': body_second}


def _subject_line_issue(subject_line: str) -> str | None:
    if not subject_line:
        return None
    if '[' in subject_line or ']' in subject_line:
        return 'Contains bracket artifact'
    if ';' in subject_line:
        return 'Contains semicolon artifact'
    if TIME_ARTIFACT_RE.search(subject_line):
        return 'Contains time artifact'
    return None


def validate_attendance_pdf(pdf_path: str | Path) -> Dict[str, List[str]]:
    pdf_path = Path(pdf_path)
    issues: Dict[str, List[str]] = {
        'empty_first_row': [],
        'subject_artifacts': [],
        'name_number_artifacts': [],
    }

    with pdfplumber.open(str(pdf_path)) as pdf:
        for page_no, page in enumerate(pdf.pages, start=1):
            text = page.extract_text() or ''
            lines = [ln.strip() for ln in text.splitlines() if ln.strip()]

            # 1) Empty first row check
            empty_first = _detect_empty_first_body_row(page)
            if empty_first:
                issues['empty_first_row'].append(
                    f"Page {page_no}: row1_y={empty_first['row1_y']:.2f}, top={empty_first['top_line']:.2f}, second={empty_first['second_line']:.2f}"
                )

            # 2) Subject artifact check
            subject_line = next((ln for ln in lines if ln.lower().startswith('subject:')), '')
            subject_issue = _subject_line_issue(subject_line)
            if subject_issue:
                issues['subject_artifacts'].append(f'Page {page_no}: {subject_issue} -> {subject_line}')

            # 3) Number artifact inside names
            for ln in lines:
                m = STUDENT_LINE_RE.match(ln)
                if not m:
                    continue
                serial = int(m.group(1))
                if serial < 1 or serial > 60:
                    continue
                name_part = m.group(2).strip()
                if NAME_DIGIT_RE.search(name_part):
                    issues['name_number_artifacts'].append(f'Page {page_no}: {ln}')

    return issues


def validate_generated_outputs(output_dir: str | Path, grades: List[int] | None = None) -> Dict[int, Dict[str, List[str]]]:
    output_dir = Path(output_dir)
    grade_list = sorted({int(g) for g in grades}) if grades else [9, 10, 11, 12]

    report: Dict[int, Dict[str, List[str]]] = {}
    print('Validation Summary')

    for g in grade_list:
        pdf_path = output_dir / f'Grade_{g}_Attendance.pdf'
        if not pdf_path.exists():
            print(f'- Grade {g}: MISSING ({pdf_path})')
            continue

        issues = validate_attendance_pdf(pdf_path)
        report[g] = issues

        total = sum(len(v) for v in issues.values())
        if total == 0:
            print(f'- Grade {g}: OK')
            continue

        print(f'- Grade {g}: {total} issue(s)')
        for key, vals in issues.items():
            if vals:
                print(f'  {key}: {len(vals)}')
                for item in vals[:3]:
                    print(f'    - {item}')

    return report




In [8]:
schedule_pdf = r"C:\Eni data\PYTHON\Exam_Schedule_Clean.pdf"
template_pdf = r"C:\Eni data\PYTHON\Grade 10 Attendance sheet sample.pdf"

# Use standalone student PDFs by grade.
students_pdf = {
    9: r"C:\Eni data\PYTHON\Grade 9 Student names list.pdf",
    11: r"C:\Eni data\PYTHON\Grade 11 Student names list.pdf",
    12: r"C:\Eni data\PYTHON\Grade 12 Student names list.pdf",
}

output_dir = r"C:\Eni data\PYTHON\ML\output"
grades_only = [9, 11, 12]
year_override = None
include_times = False

generate_attendance_pdfs(
    schedule_pdf=schedule_pdf,
    students_pdf=students_pdf,
    template_pdf=template_pdf,
    output_dir=output_dir,
    grades=grades_only,
    year=year_override,
    include_times=include_times,
)

validation_report = validate_generated_outputs(output_dir=output_dir, grades=grades_only)



Generated C:\Eni data\PYTHON\ML\output\Grade_9_Attendance.pdf
Generated C:\Eni data\PYTHON\ML\output\Grade_11_Attendance.pdf
Generated C:\Eni data\PYTHON\ML\output\Grade_12_Attendance.pdf
Validation Summary
- Grade 9: OK
- Grade 11: OK
- Grade 12: OK
